In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Tuple, Dict, Set
from abc import abstractmethod, ABC
import random
import numpy as np
import pandas as pd
import copy
import pickle
import itertools
import string



In [ ]:
from dataset_utils.paths import get_dataset_dir
from dataset_utils.utils import file_handle, grouping
from dataset_utils.utils.tree import Node

dataset_dir = get_dataset_dir()
generated_datasets_dir = dataset_dir / "datasets"

random.seed(420)
np.random.seed(420)


In [ ]:
from dataset_utils.reader import ADIQDataset
from dataset_utils.creation import save_dataset
from dataset_utils.question import Question
from dataset_utils.outputs import question_formatting, to_basic_prompt

# The released DiagnosticIQ Pert V3.1 was perturbed from DiagnosticIQ (4 options);
# use "DiagnosticIQ" here to reproduce it.
source_dataset = "DiagnosticIQPro"
ds = ADIQDataset(generated_datasets_dir / source_dataset)

In [ ]:
def _add_left_parenthesis(s: str) -> str:
    return f"({s}"


def _add_left_bracket(s: str) -> str:
    return f"[{s}"


def _add_left_brace(s: str) -> str:
    return f"{{{s}"


def _add_left_wave(s: str) -> str:
    return f"~{s}"


def _add_right_parenthesis(s: str) -> str:
    return f"{s})"


def _add_right_bracket(s: str) -> str:
    return f"{s}]"


def _add_right_brace(s: str) -> str:
    return f"{s}}}"


def _add_right_wave(s: str) -> str:
    return f"{s}~"


def _add_right_eq(s: str) -> str:
    return f"{s}="


def _add_parentheses(s: str) -> str:
    return f"({s})"


def _add_brackets(s: str) -> str:
    return f"[{s}]"


def _add_braces(s: str) -> str:
    return f"{{{s}}}"


def _add_waves(s: str) -> str:
    return f"~{s}~"


def _caesar(s: str, delta: int = 10) -> str:
    return chr(ord(s) + delta)

In [ ]:
from functools import partial
from dataset_utils.creation import selection_question_library, elimination_question_library

class KPPerturbation(ABC):
    def __init__(self):
        self.method = "default"
        pass

    @abstractmethod
    def perturb(self, mcq: Question) -> Question:
        pass


class OptionFormatPerturbation(KPPerturbation):
    def __init__(
        self,
        method: str = "add_parentheses",
    ):
        """
        Args:
            method:str, the perturbation method
        """
        super().__init__()
        self.method = method
        if method == "add_left_parenthesis":
            self.formatter = _add_left_parenthesis
        elif method == "add_left_bracket":
            self.formatter = _add_left_bracket
        elif method == "add_left_brace":
            self.formatter = _add_left_brace
        elif method == "add_left_wave":
            self.formatter = _add_left_wave
        elif method == "add_right_parenthesis":
            self.formatter = _add_right_parenthesis
        elif method == "add_right_bracket":
            self.formatter = _add_right_bracket
        elif method == "add_right_brace":
            self.formatter = _add_right_brace
        elif method == "add_right_wave":
            self.formatter = _add_right_wave
        elif method == "add_right_eq":
            self.formatter = _add_right_eq
        elif method == "add_parentheses":
            self.formatter = _add_parentheses
        elif method == "add_brackets":
            self.formatter = _add_brackets
        elif method == "add_braces":
            self.formatter = _add_braces
        elif method == "add_waves":
            self.formatter = _add_waves
        else:
            raise Exception("Invalid option format perturbation method.")

    def __str__(self):
        return f"OptionFormatPerturbation.{self.method}"

    def perturb(self, mcq: Question) -> Question:
        assert len(mcq.option_ids) == len(mcq.options)
        try:
            new_option_ids = [self.formatter(elem) for elem in mcq.option_ids]
            result = copy.deepcopy(mcq)
            result.option_ids = new_option_ids
        except:
            print("OptionFormatPerturbation error. Keep the original result.")
            result = copy.deepcopy(mcq)
        return result
    
class CaesarPerturbation(KPPerturbation):
    def __init__(self, delta: int = 20):
        """
        Args:
            delta:int, the offset value in ASCII of option ids.
        """
        super().__init__()
        self.formatter = partial(_caesar, delta=delta)

    def __str__(self):
        return f"CaesarPerturbation.{self.method}"

    def perturb(self, mcq: Question) -> Question:
        assert len(mcq.option_ids) == len(mcq.options)

        try:
            new_option_ids = [self.formatter(elem) for elem in mcq.option_ids]
            result = copy.deepcopy(mcq)
            result.option_ids = new_option_ids
        except:
            print("CaesarPerturbation error. Keep the original result.")
            result = copy.deepcopy(mcq)
        return result
    
class QuestionPromptPerturbation(KPPerturbation):
    def __init__(self):
        """
        Args:
            delta:int, the offset value in ASCII of option ids.
        """
        super().__init__()

    def __str__(self):
        return f"QuestionPromptPerturbation.{self.method}"

    def perturb(self, mcq: Question) -> Question:
        assert len(mcq.option_ids) == len(mcq.options)

        try:
            result = copy.deepcopy(mcq)
            if result.question_type == "positive":
                result.question_prompt = random.choice(list(set(selection_question_library) - {result.question_prompt}))
            else:
                result.question_prompt = random.choice(list(set(elimination_question_library) - {result.question_prompt}))
        except:
            print("QuestionPromptPerturbation error. Keep the original result.")
            result = copy.deepcopy(mcq)
        return result
    
class RandomizeCondOptionPerturbation(KPPerturbation):
    def __init__(self):
        """
        Args:
            delta:int, the offset value in ASCII of option ids.
        """
        super().__init__()

    def __str__(self):
        return f"RandomizeCondOptionPerturbation.{self.method}"

    def perturb(self, mcq: Question) -> Question:
        assert len(mcq.option_ids) == len(mcq.options)


        try:
            result = copy.deepcopy(mcq)
            random.shuffle(result.condition_description)
            
            _ids = list(range(len(result.options)))
            random.shuffle(_ids)

            result.options = [result.options[x] for x in _ids]
            result.option_ids = [result.option_ids[x] for x in _ids]
            result.correct = [result.correct[x] for x in _ids]
            
        except:
            print("RandomizeCondOptionPerturbation error. Keep the original result.")
            result = copy.deepcopy(mcq)
        return result
    


In [ ]:
option_id_pert = CaesarPerturbation(delta=15)
paranthesis_pert = OptionFormatPerturbation()
qp_pert = QuestionPromptPerturbation()
rand_cond_obs_pert = RandomizeCondOptionPerturbation()


In [ ]:
simple_pert_ques:List[Question] = []

for q in ds.questions:
    q1 = option_id_pert.perturb(q)
    q2 = paranthesis_pert.perturb(q1)
    q3 = qp_pert.perturb(q2)
    q4 = rand_cond_obs_pert.perturb(q3)

    simple_pert_ques.append(q4)

ds.questions = simple_pert_ques


## Question formatting

Writes the final question text (asset description, conditions, duration and prompt) and sets `question_first`/`text_type`. In the source repository this ran as a separate notebook, `question_structuring.ipynb`, over the saved dataset.

Re-applied here because the perturbations shuffle the conditions and replace the prompt.

In [ ]:
for q in ds.questions:
    question_formatting(q, ds.asset_descriptions)

In [ ]:
ds.save(
    "DiagnosticIQPert",
    generated_datasets_dir
)